In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [ ]:
file_path = '/Users/omeremeksiz/Desktop/master/research/deep-cs-prediction/src/cs_conformer/labels/ssq_original.xlsx'
data = pd.read_excel(file_path, header=None)

In [ ]:
values = data.iloc[:, 1:].values.flatten().astype(float)
values = values.reshape(-1, 1)
scaler = StandardScaler()
scaled_data = scaler.fit_transform(values)
print("Number of label:",len(values))

In [ ]:
inertia = []
k_values = range(1, 20)

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(scaled_data)
    inertia.append(kmeans.inertia_)

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(k_values, inertia, marker='o')
plt.title('Elbow Method: k vs WCSS')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Within Cluster Sum of Squares (WCSS)')
plt.grid(True)
plt.show()

In [ ]:
optimal_k = 4  
kmeans = KMeans(n_clusters=optimal_k, random_state=42)
kmeans.fit(scaled_data)

clusters = kmeans.labels_
original_data = pd.DataFrame(values, columns=['Values'])
original_data['Cluster'] = clusters

sorted_cluster_centers = np.argsort(scaler.inverse_transform(kmeans.cluster_centers_).flatten())
cluster_mapping = {old: new for new, old in enumerate(sorted_cluster_centers)}
original_data['Cluster'] = original_data['Cluster'].map(cluster_mapping)

cluster_centers = sorted(scaler.inverse_transform(kmeans.cluster_centers_).flatten())
thresholds = [
    (cluster_centers[i] + cluster_centers[i + 1]) / 2 for i in range(len(cluster_centers) - 1)
]

plt.figure(figsize=(10, 6))
for cluster in range(optimal_k):
    cluster_data = original_data[original_data['Cluster'] == cluster]['Values']
    plt.hist(cluster_data, bins=20, alpha=0.6, label=f'Cluster {cluster + 1}')

for idx, threshold in enumerate(thresholds, start=1):
    plt.axvline(x=threshold, color='k', linestyle='--', linewidth=1, label=f'Threshold {idx}: {threshold:.2f}')

plt.title('Clusters Visualization (Histogram)')
plt.xlabel('Values')
plt.ylabel('Frequency')
plt.grid(True)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()  
plt.show()

plt.figure(figsize=(10, 6))
colors = ['r', 'g', 'b', 'c', 'm', 'y', 'k']  
for cluster in range(optimal_k):
    cluster_data = original_data[original_data['Cluster'] == cluster]
    plt.scatter(cluster_data.index, cluster_data['Values'], alpha=0.6, label=f'Cluster {cluster + 1}', color=colors[cluster % len(colors)])

for idx, threshold in enumerate(thresholds, start=1):
    plt.axvline(x=threshold, color='k', linestyle='--', linewidth=1, label=f'Threshold {idx}: {threshold:.2f}')

plt.title('Clusters Visualization (Scatter)')
plt.xlabel('Sample Index')
plt.ylabel('Values')
plt.grid(True)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()  
plt.show()

# Print thresholds and cluster sizes
cluster_sizes = original_data['Cluster'].value_counts().sort_index()
threshold_intervals = [
    f'< {round(thresholds[0], 2)}'
] + [
    f'{round(thresholds[i], 2)} - {round(thresholds[i + 1], 2)}' for i in range(len(thresholds) - 1)
] + [
    f'> {round(thresholds[-1], 2)}'
]

# Ensure the number of cluster sizes matches the intervals
cluster_sizes_full = cluster_sizes.tolist() + [0] * (len(threshold_intervals) - len(cluster_sizes))

summary_table = pd.DataFrame({
    'Thresholds (Between Clusters)': threshold_intervals,
    'Cluster Size': cluster_sizes_full
})

# Print the table
print(summary_table.to_string(index=False))

In [ ]:
file_path = '/Users/omeremeksiz/Desktop/master/research/deep-cs-prediction/src/cs_conformer/labels/ssq_original.xlsx'
data = pd.read_excel(file_path, header=None)
values = data.iloc[:, 1:].values.flatten().astype(float)
values = values.reshape(-1, 1)
scaler = StandardScaler()
scaled_data = scaler.fit_transform(values)

def custom_kmeans(data, n_clusters, max_iter=300, tol=1e-4, alpha=0.5):
    np.random.seed(42)
    n_samples = data.shape[0]
    initial_indices = np.random.choice(n_samples, n_clusters, replace=False)
    centers = data[initial_indices]

    for iteration in range(max_iter):
        distances = np.linalg.norm(data[:, np.newaxis] - centers, axis=2)
        labels = np.argmin(distances, axis=1)

        wcss = sum(np.min(distances, axis=1)**2)

        cluster_sizes = np.array([np.sum(labels == i) for i in range(n_clusters)])
        size_penalty = np.var(cluster_sizes)

        total_loss = alpha * wcss + (1 - alpha) * size_penalty

        new_centers = np.array([data[labels == i].mean(axis=0) if np.sum(labels == i) > 0 else centers[i]
                                for i in range(n_clusters)])
        
        if np.linalg.norm(new_centers - centers) < tol:
            break
        centers = new_centers

    return labels, centers, wcss, size_penalty, total_loss

def compute_loss_for_k(data, max_k, alpha=0.5):
    losses = []
    for k in range(1, max_k + 1):
        labels, centers, wcss, size_penalty, total_loss = custom_kmeans(data, n_clusters=k, alpha=alpha)
        losses.append(total_loss)
    return losses

max_k = 20
alpha = 0.5  #
losses = compute_loss_for_k(scaled_data, max_k, alpha)

optimal_k = np.argmin(losses) + 1

# Loss vs K plot
plt.figure(figsize=(10, 6))
plt.plot(range(1, max_k + 1), losses, marker='o', linewidth=2.5, markersize=8, label='Kayıp')
plt.scatter(optimal_k, losses[optimal_k - 1], color='red', label=f'Optimal k = {optimal_k}', s=120, zorder=5)

# Font and axis enhancements
plt.xlabel('Küme Sayısı (k)', fontsize=14, fontweight='bold')
plt.ylabel('Kayıp', fontsize=14, fontweight='bold')
plt.xticks(fontsize=12, fontweight='bold')
plt.yticks(fontsize=12, fontweight='bold')
plt.legend(fontsize=12, frameon=True, fancybox=True, framealpha=0.9, borderpad=1, loc='best')
for text in plt.gca().get_legend().get_texts():
    text.set_fontweight('bold')
    
plt.grid(True)

# Save the plot
save_path = '/Users/omeremeksiz/Desktop/master/research/deep-cs-prediction/src/cs_conformer/preprocessing/optimal_k_plot.png'
plt.tight_layout()
plt.savefig(save_path, dpi=300)
plt.show()

labels, centers, wcss, size_penalty, total_loss = custom_kmeans(scaled_data, n_clusters=optimal_k, alpha=alpha)

original_data = pd.DataFrame(values, columns=['Values'])
original_data['Cluster'] = labels

sorted_cluster_centers = np.argsort(scaler.inverse_transform(centers).flatten())
cluster_mapping = {old: new for new, old in enumerate(sorted_cluster_centers)}
original_data['Cluster'] = original_data['Cluster'].map(cluster_mapping)

cluster_centers = sorted(scaler.inverse_transform(centers).flatten())
thresholds = [
    (cluster_centers[i] + cluster_centers[i + 1]) / 2 for i in range(len(cluster_centers) - 1)
]

# Histogram plot
plt.figure(figsize=(10, 6))
for cluster in range(optimal_k):
    cluster_data = original_data[original_data['Cluster'] == cluster]['Values']
    plt.hist(cluster_data, bins=20, alpha=0.6, label=f'Cluster {cluster + 1}')

for idx, threshold in enumerate(thresholds, start=1):
    plt.axvline(x=threshold, color='k', linestyle='--', linewidth=1, label=f'Threshold {idx}: {threshold:.2f}')

plt.title('Clusters Visualization (Histogram)')
plt.xlabel('Values')
plt.ylabel('Frequency')
plt.grid(True)

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()  
plt.show()

# Scatter plot
plt.figure(figsize=(10, 6))
colors = ['r', 'g', 'b', 'c', 'm', 'y', 'k']
for cluster in range(optimal_k):
    cluster_data = original_data[original_data['Cluster'] == cluster]
    plt.scatter(
        cluster_data.index,
        cluster_data['Values'],
        alpha=0.6,
        color=colors[cluster % len(colors)]
    )

# Threshold lines
for threshold in thresholds:
    plt.axhline(y=threshold, color='k', linestyle='--', linewidth=1)

# Styled fonts (same as example)
plt.xlabel('Sample Index', fontsize=20, fontweight='bold')
plt.ylabel('Values', fontsize=20, fontweight='bold')
plt.xticks(fontsize=18, fontweight='bold')
plt.yticks(fontsize=18, fontweight='bold')

plt.grid(True)

# Save the plot
save_path = '/Users/omeremeksiz/Desktop/master/research/deep-cs-prediction/src/cs_conformer/preprocessing/cluster_scatter.png'
plt.tight_layout()
plt.savefig(save_path, dpi=300)
plt.show()

# Cluster size vs cluster index
cluster_sizes = original_data['Cluster'].value_counts().sort_index()

plt.figure(figsize=(10, 6))
plt.bar(cluster_sizes.index, cluster_sizes.values, color='skyblue')

# Font styling (like your second example)
plt.xlabel('Cluster', fontsize=20, fontweight='bold')
plt.ylabel('Size', fontsize=20, fontweight='bold')
plt.xticks(fontsize=18, fontweight='bold')
plt.yticks(fontsize=18, fontweight='bold')

plt.grid(axis='y')

# Save the plot
save_path = '/Users/omeremeksiz/Desktop/master/research/deep-cs-prediction/src/cs_conformer/preprocessing/cluster_histogram.png'
plt.tight_layout()
plt.savefig(save_path, dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Histogram of original (pre-clustering) labels
plt.figure(figsize=(10, 6))
plt.hist(original_data['Values'], bins=30, color='green', edgecolor='black')

# Font styling (same as cluster bar plot)
plt.xlabel('Küme', fontsize=20, fontweight='bold')
plt.ylabel('Veri Sayısı', fontsize=20, fontweight='bold')
plt.xticks(fontsize=18, fontweight='bold')
plt.yticks(fontsize=18, fontweight='bold')

plt.grid(axis='y')

# Save the plot
save_path = '/Users/omeremeksiz/Desktop/master/research/deep-cs-prediction/src/cs_conformer/preprocessing/original_histogram.png'
plt.tight_layout()
plt.savefig(save_path, dpi=300)
plt.show()
